# Machine Learning Lab — 10 Practicals
**Course:** U24AIMLPC508 | **Semester:** V  
**Student:** Akash Kumar Saw | **Roll No.:** 03  
**University:** D. Y. Patil University, Pune, Ambi

This notebook follows the 10 practicals in the supplied Machine Learning Lab Manual. It uses datasets bundled with `scikit-learn` wherever the experiment permits. The logic-gate experiment necessarily uses the standard truth table rather than a real-world dataset.

**Reproducibility:** fixed random seeds are used for train/test splits and estimators where applicable. Run the notebook from top to bottom.

## Environment and imports

The datasets used below are available through scikit-learn's dataset loaders, so no external dataset download is required. The notebook prints measured outputs from the executed code rather than relying on handwritten sample output.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_iris, load_diabetes, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, accuracy_score, classification_report,
    confusion_matrix
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neural_network import MLPClassifier

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.5
NumPy: 2.3.5
pandas: 2.2.3
scikit-learn: 1.8.0


# Practical 1 — Mean, Median, Standard Deviation and Mode

**Aim:** Compute descriptive statistics using user-defined functions.  
**Dataset:** Iris sepal-length measurements from `sklearn.datasets.load_iris`. The measurements are rounded to one decimal place for frequency-based mode calculation; the original dataset remains the source.

In [2]:
iris = load_iris(as_frame=True)
sepal_length = iris.data["sepal length (cm)"].round(1).tolist()

from collections import Counter

def user_mean(values):
    if len(values) == 0:
        raise ValueError("The dataset must not be empty.")
    return sum(values) / len(values)

def user_median(values):
    if len(values) == 0:
        raise ValueError("The dataset must not be empty.")
    ordered = sorted(values)
    n = len(ordered)
    middle = n // 2
    if n % 2:
        return ordered[middle]
    return (ordered[middle - 1] + ordered[middle]) / 2

def user_population_std(values):
    if len(values) == 0:
        raise ValueError("The dataset must not be empty.")
    avg = user_mean(values)
    variance = sum((x - avg) ** 2 for x in values) / len(values)
    return variance ** 0.5

def user_modes(values):
    if len(values) == 0:
        raise ValueError("The dataset must not be empty.")
    counts = Counter(values)
    highest = max(counts.values())
    return sorted([value for value, count in counts.items() if count == highest])

print("Dataset: Iris sepal length, rounded to 1 decimal place")
print("Number of observations:", len(sepal_length))
print("First 10 values:", sepal_length[:10])
print(f"Mean: {user_mean(sepal_length):.4f}")
print(f"Median: {user_median(sepal_length):.4f}")
print(f"Population standard deviation: {user_population_std(sepal_length):.4f}")
print("Mode(s):", user_modes(sepal_length))

Dataset: Iris sepal length, rounded to 1 decimal place
Number of observations: 150
First 10 values: [5.1, 4.9, 4.7, 4.6, 5.0, 5.4, 4.6, 5.0, 4.4, 4.9]
Mean: 5.8433
Median: 5.8000
Population standard deviation: 0.8253
Mode(s): [5.0]


**Note:** The manual's standard-deviation formula divides by *N*, so this experiment reports population standard deviation. Pandas' default `std()` in Practical 2 uses sample standard deviation (*N − 1*); the distinction is intentional.

In [3]:
# Small edge-case checks for the user-defined functions
assert user_median([5, 7, 9]) == 7
assert user_median([4, 6, 8, 10]) == 7
assert user_modes([1, 1, 2, 2, 3]) == [1, 2]
print("Validation checks passed: odd median, even median, and multimodal data.")

Validation checks passed: odd median, even median, and multimodal data.


# Practical 2 — DataFrame, Column Retrieval and Summary Statistics

**Aim:** Create a DataFrame with at least five observations, retrieve a column, summarise the data, and calculate mean and standard deviation.  
**Dataset:** First 10 observations from scikit-learn's Iris dataset.

In [4]:
iris = load_iris(as_frame=True)
df = iris.frame.head(10).copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df = df.drop(columns=["target"])

print("DataFrame (first 10 Iris observations):")
display(df)
print("\nRetrieved column — sepal length (cm):")
display(df["sepal length (cm)"])
print("\nSummary statistics:")
display(df.describe())
print("\nMean of numeric columns:")
display(df.mean(numeric_only=True))
print("\nSample standard deviation of numeric columns:")
display(df.std(numeric_only=True))

DataFrame (first 10 Iris observations):


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
5,5.4,3.9,1.7,0.4,setosa
6,4.6,3.4,1.4,0.3,setosa
7,5.0,3.4,1.5,0.2,setosa
8,4.4,2.9,1.4,0.2,setosa
9,4.9,3.1,1.5,0.1,setosa



Retrieved column — sepal length (cm):


0    5.1
1    4.9
2    4.7
3    4.6
4    5.0
5    5.4
6    4.6
7    5.0
8    4.4
9    4.9
Name: sepal length (cm), dtype: float64


Summary statistics:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,10.000000,10.000000,10.000000,10.000000
mean,4.860000,3.310000,1.450000,0.220000
std,0.291357,0.307137,0.108012,0.078881
min,4.400000,2.900000,1.300000,0.100000
25%,4.625000,3.100000,1.400000,0.200000
50%,4.900000,3.300000,1.400000,0.200000
75%,5.000000,3.475000,1.500000,0.200000
max,5.400000,3.900000,1.700000,0.400000



Mean of numeric columns:


sepal length (cm)    4.86
sepal width (cm)     3.31
petal length (cm)    1.45
petal width (cm)     0.22
dtype: float64


Sample standard deviation of numeric columns:


sepal length (cm)    0.291357
sepal width (cm)     0.307137
petal length (cm)    0.108012
petal width (cm)     0.078881
dtype: float64

# Practical 3 — Linear Regression with CSV Data and Mean Squared Error

**Aim:** Fit linear regression using a CSV training file and evaluate it on held-out test data.  
**Dataset:** Real diabetes dataset bundled with scikit-learn. To keep this a simple linear-regression experiment, BMI is used as the single input feature and disease-progression measure as the target. The data is split first; training and test partitions are written to CSV and read back.

In [5]:
diabetes = load_diabetes(as_frame=True)
reg_df = diabetes.frame[["bmi", "target"]].rename(columns={"bmi": "X", "target": "Y"})
train_df, test_df = train_test_split(reg_df, test_size=0.25, random_state=42)

train_csv = "linear_train_diabetes.csv"
test_csv = "linear_test_diabetes.csv"
train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)
train_data = pd.read_csv(train_csv)
test_data = pd.read_csv(test_csv)

X_train = train_data[["X"]]
y_train = train_data["Y"]
X_test = test_data[["X"]]
y_test = test_data["Y"]

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)
linear_mse = mean_squared_error(y_test, linear_predictions)

print("Dataset: sklearn diabetes")
print("Training rows:", len(train_data), "| Test rows:", len(test_data))
print(f"Slope: {linear_model.coef_[0]:.4f}")
print(f"Intercept: {linear_model.intercept_:.4f}")
print("First 10 actual values:     ", np.round(y_test.to_numpy()[:10], 2))
print("First 10 predicted values:  ", np.round(linear_predictions[:10], 2))
print(f"Test Mean Squared Error: {linear_mse:.4f}")

Dataset: sklearn diabetes
Training rows: 331 | Test rows: 111
Slope: 975.2770
Intercept: 152.0765
First 10 actual values:      [219.  70. 202. 230. 111.  84. 242. 272.  94.  96.]
First 10 predicted values:   [146.02 188.07 148.13 202.79 132.36 128.15 318.42 196.48  64.03 167.05]
Test Mean Squared Error: 3775.6172


# Practical 4 — Non-linear (Polynomial) Regression with CSV Data

**Aim:** Implement polynomial regression and evaluate it with Mean Squared Error.  
**Dataset:** The same real scikit-learn diabetes dataset, using BMI as the input and target as the response. Degree-2 polynomial features allow a curved relationship. Train/test split and CSV loading follow Practical 3.

In [6]:
diabetes = load_diabetes(as_frame=True)
poly_df = diabetes.frame[["bmi", "target"]].rename(columns={"bmi": "X", "target": "Y"})
train_df, test_df = train_test_split(poly_df, test_size=0.25, random_state=42)

train_csv = "nonlinear_train_diabetes.csv"
test_csv = "nonlinear_test_diabetes.csv"
train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)
train_data = pd.read_csv(train_csv)
test_data = pd.read_csv(test_csv)

poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)
poly_model.fit(train_data[["X"]], train_data["Y"])
poly_predictions = poly_model.predict(test_data[["X"]])
poly_mse = mean_squared_error(test_data["Y"], poly_predictions)

print("Dataset: sklearn diabetes")
print("Polynomial degree: 2")
print("Training rows:", len(train_data), "| Test rows:", len(test_data))
print("First 10 actual values:     ", np.round(test_data["Y"].to_numpy()[:10], 2))
print("First 10 predicted values:  ", np.round(poly_predictions[:10], 2))
print(f"Test Mean Squared Error: {poly_mse:.4f}")

Dataset: sklearn diabetes
Polynomial degree: 2
Training rows: 331 | Test rows: 111
First 10 actual values:      [219.  70. 202. 230. 111.  84. 242. 272.  94.  96.]
First 10 predicted values:   [144.31 186.31 146.33 201.79 131.38 127.47 337.64 195.11  71.98 164.89]
Test Mean Squared Error: 3803.6562


# Practical 5 — Logistic Regression and Classification Accuracy

**Aim:** Train a Logistic Regression classifier from a CSV file and compute accuracy.  
**Dataset:** scikit-learn Breast Cancer Wisconsin dataset. The target is binary: malignant vs. benign. A stratified split preserves class proportions.

In [7]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Demonstrate the CSV-based workflow requested in the manual.
train_class_df = X_train.copy()
train_class_df["target"] = y_train
test_class_df = X_test.copy()
test_class_df["target"] = y_test
train_class_df.to_csv("logistic_train_breast_cancer.csv", index=False)
test_class_df.to_csv("logistic_test_breast_cancer.csv", index=False)
train_class_df = pd.read_csv("logistic_train_breast_cancer.csv")
test_class_df = pd.read_csv("logistic_test_breast_cancer.csv")

X_train = train_class_df.drop(columns="target")
y_train = train_class_df["target"]
X_test = test_class_df.drop(columns="target")
y_test = test_class_df["target"]

logistic_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=42))
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)

print("Dataset: Breast Cancer Wisconsin (sklearn)")
print("Class names:", list(cancer.target_names), "(0 =", cancer.target_names[0], ", 1 =", cancer.target_names[1], ")")
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print(f"Accuracy: {accuracy_score(y_test, logistic_predictions):.4f}")
print("\nConfusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, logistic_predictions))
print("\nClassification report:")
print(classification_report(y_test, logistic_predictions, target_names=cancer.target_names, zero_division=0))

Dataset: Breast Cancer Wisconsin (sklearn)
Class names: [np.str_('malignant'), np.str_('benign')] (0 = malignant , 1 = benign )
Training rows: 426 | Test rows: 143
Accuracy: 0.9860

Confusion matrix (rows = actual, columns = predicted):
[[52  1]
 [ 1 89]]

Classification report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        53
      benign       0.99      0.99      0.99        90

    accuracy                           0.99       143
   macro avg       0.99      0.99      0.99       143
weighted avg       0.99      0.99      0.99       143



# Practical 6 — Naive Bayes Classifier and Accuracy

**Aim:** Implement a Naive Bayes classifier from CSV data and evaluate held-out accuracy.  
**Dataset:** scikit-learn Wine recognition dataset. Gaussian Naive Bayes is used because the features are continuous numeric measurements and the task has three classes.

In [8]:
wine = load_wine(as_frame=True)
X = wine.data
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

train_nb = X_train.copy()
train_nb["target"] = y_train
test_nb = X_test.copy()
test_nb["target"] = y_test
train_nb.to_csv("naive_bayes_train_wine.csv", index=False)
test_nb.to_csv("naive_bayes_test_wine.csv", index=False)
train_nb = pd.read_csv("naive_bayes_train_wine.csv")
test_nb = pd.read_csv("naive_bayes_test_wine.csv")

X_train = train_nb.drop(columns="target")
y_train = train_nb["target"]
X_test = test_nb.drop(columns="target")
y_test = test_nb["target"]

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)

print("Dataset: sklearn Wine recognition")
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print(f"Accuracy: {accuracy_score(y_test, nb_predictions):.4f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, nb_predictions))
print("\nClassification report:")
print(classification_report(y_test, nb_predictions, target_names=wine.target_names, zero_division=0))

Dataset: sklearn Wine recognition
Training rows: 133 | Test rows: 45
Accuracy: 0.9778

Confusion matrix:
[[15  0  0]
 [ 1 17  0]
 [ 0  0 12]]

Classification report:
              precision    recall  f1-score   support

     class_0       0.94      1.00      0.97        15
     class_1       1.00      0.94      0.97        18
     class_2       1.00      1.00      1.00        12

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45



# Practical 7 — k-Nearest Neighbors on the Iris Dataset

**Aim:** Classify Iris observations using k-NN and print correct and incorrect predictions.  
**Dataset:** scikit-learn Iris (150 observations, four features, three species). The split and k=3 follow the manual.

In [9]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.30, random_state=42, stratify=iris.target
)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
knn.fit(X_train, y_train)
knn_predictions = knn.predict(X_test)

correct_count = 0
wrong_count = 0
for i, (actual, predicted) in enumerate(zip(y_test, knn_predictions), start=1):
    status = "Correct" if actual == predicted else "Wrong"
    correct_count += int(actual == predicted)
    wrong_count += int(actual != predicted)
    print(
        f"{i:02d}. Actual: {iris.target_names[actual]:10s} | "
        f"Predicted: {iris.target_names[predicted]:10s} | {status}"
    )

print("\nCorrect predictions:", correct_count)
print("Wrong predictions:", wrong_count)
print(f"Accuracy: {accuracy_score(y_test, knn_predictions):.4f}")

01. Actual: virginica  | Predicted: virginica  | Correct
02. Actual: versicolor | Predicted: versicolor | Correct
03. Actual: virginica  | Predicted: versicolor | Wrong
04. Actual: versicolor | Predicted: versicolor | Correct
05. Actual: virginica  | Predicted: virginica  | Correct
06. Actual: virginica  | Predicted: virginica  | Correct
07. Actual: versicolor | Predicted: versicolor | Correct
08. Actual: versicolor | Predicted: versicolor | Correct
09. Actual: setosa     | Predicted: setosa     | Correct
10. Actual: virginica  | Predicted: virginica  | Correct
11. Actual: setosa     | Predicted: setosa     | Correct
12. Actual: setosa     | Predicted: setosa     | Correct
13. Actual: virginica  | Predicted: virginica  | Correct
14. Actual: virginica  | Predicted: virginica  | Correct
15. Actual: setosa     | Predicted: setosa     | Correct
16. Actual: virginica  | Predicted: virginica  | Correct
17. Actual: versicolor | Predicted: versicolor | Correct
18. Actual: setosa     | Predicte

# Practical 8 — ID3-Style Decision Tree (Entropy / Information Gain)

**Aim:** Demonstrate an entropy-based decision tree and classify a new sample.  
**Dataset:** scikit-learn Iris. The manual describes ID3 using categorical Play Tennis data; here, to meet the requirement to use a built-in real dataset, the entropy criterion is demonstrated on numeric Iris measurements. `DecisionTreeClassifier(criterion="entropy")` uses information gain, making this an ID3-style demonstration rather than a claim that scikit-learn implements the original categorical ID3 algorithm exactly.

In [10]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

id3_style = DecisionTreeClassifier(criterion="entropy", random_state=42)
id3_style.fit(X_train, y_train)
predictions = id3_style.predict(X_test)

print("Dataset: sklearn Iris")
print("Criterion: entropy (information gain)")
print(f"Test accuracy: {accuracy_score(y_test, predictions):.4f}")
print("\nTree rules (truncated to depth 3 for readability):")
print(export_text(id3_style, feature_names=list(X.columns), max_depth=3))

new_sample = pd.DataFrame([{
    "sepal length (cm)": 5.1,
    "sepal width (cm)": 3.5,
    "petal length (cm)": 1.4,
    "petal width (cm)": 0.2
}])
new_prediction = id3_style.predict(new_sample)[0]
print("New sample:", new_sample.to_dict(orient="records")[0])
print("Predicted class:", iris.target_names[new_prediction])

Dataset: sklearn Iris
Criterion: entropy (information gain)
Test accuracy: 0.8889

Tree rules (truncated to depth 3 for readability):
|--- petal length (cm) <= 2.45
|   |--- class: 0
|--- petal length (cm) >  2.45
|   |--- petal width (cm) <= 1.55
|   |   |--- petal length (cm) <= 4.95
|   |   |   |--- class: 1
|   |   |--- petal length (cm) >  4.95
|   |   |   |--- class: 2
|   |--- petal width (cm) >  1.55
|   |   |--- sepal length (cm) <= 6.10
|   |   |   |--- petal width (cm) <= 1.85
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- petal width (cm) >  1.85
|   |   |   |   |--- class: 2
|   |   |--- sepal length (cm) >  6.10
|   |   |   |--- class: 2

New sample: {'sepal length (cm)': 5.1, 'sepal width (cm)': 3.5, 'petal length (cm)': 1.4, 'petal width (cm)': 0.2}
Predicted class: setosa


# Practical 9 — CART Decision Tree (Gini Impurity)

**Aim:** Demonstrate a CART-style decision tree using Gini impurity and classify a new sample.  
**Dataset:** scikit-learn Wine recognition. `DecisionTreeClassifier(criterion="gini")` learns binary splits using Gini impurity.

In [11]:
wine = load_wine(as_frame=True)
X = wine.data
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

cart = DecisionTreeClassifier(criterion="gini", random_state=42)
cart.fit(X_train, y_train)
cart_predictions = cart.predict(X_test)

print("Dataset: sklearn Wine recognition")
print("Criterion: gini")
print(f"Test accuracy: {accuracy_score(y_test, cart_predictions):.4f}")
print("\nTree rules (truncated to depth 3 for readability):")
print(export_text(cart, feature_names=list(X.columns), max_depth=3))

new_sample = X.median().to_frame().T
new_prediction = cart.predict(new_sample)[0]
print("\nNew sample uses the median of each feature in the full Wine dataset.")
print("Predicted class:", wine.target_names[new_prediction])

Dataset: sklearn Wine recognition
Criterion: gini
Test accuracy: 0.9630

Tree rules (truncated to depth 3 for readability):
|--- color_intensity <= 3.82
|   |--- ash <= 3.00
|   |   |--- od280/od315_of_diluted_wines <= 3.73
|   |   |   |--- class: 1
|   |   |--- od280/od315_of_diluted_wines >  3.73
|   |   |   |--- class: 0
|   |--- ash >  3.00
|   |   |--- class: 0
|--- color_intensity >  3.82
|   |--- flavanoids <= 1.58
|   |   |--- alcalinity_of_ash <= 17.65
|   |   |   |--- class: 1
|   |   |--- alcalinity_of_ash >  17.65
|   |   |   |--- class: 2
|   |--- flavanoids >  1.58
|   |   |--- proline <= 724.50
|   |   |   |--- malic_acid <= 3.92
|   |   |   |   |--- class: 1
|   |   |   |--- malic_acid >  3.92
|   |   |   |   |--- class: 0
|   |   |--- proline >  724.50
|   |   |   |--- class: 0


New sample uses the median of each feature in the full Wine dataset.
Predicted class: class_1


# Practical 10 — Neural Network for AND, OR and XOR Logic Gates

**Aim:** Train a single-hidden-layer neural network and test logic gates.  
**Dataset note:** Logic gates are defined by truth tables, not real-world datasets, so the four standard two-input combinations are necessarily used here. A hidden layer is required for XOR because XOR is not linearly separable.

In [12]:
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=int)
gates = {
    "AND": np.array([0, 0, 0, 1]),
    "OR":  np.array([0, 1, 1, 1]),
    "XOR": np.array([0, 1, 1, 0]),
}

for gate_name, y_gate in gates.items():
    network = MLPClassifier(
        hidden_layer_sizes=(4,),
        activation="tanh",
        solver="lbfgs",
        max_iter=10000,
        random_state=42,
        tol=1e-8
    )
    network.fit(X_logic, y_gate)
    gate_predictions = network.predict(X_logic)
    print(f"\n{gate_name} Gate")
    for inputs, expected, predicted in zip(X_logic, y_gate, gate_predictions):
        status = "Correct" if expected == predicted else "Wrong"
        print(f"{inputs.tolist()} -> predicted {int(predicted)} | expected {int(expected)} | {status}")
    print("Truth table accuracy:", f"{accuracy_score(y_gate, gate_predictions):.2f}")


AND Gate
[0, 0] -> predicted 0 | expected 0 | Correct
[0, 1] -> predicted 0 | expected 0 | Correct
[1, 0] -> predicted 0 | expected 0 | Correct
[1, 1] -> predicted 1 | expected 1 | Correct
Truth table accuracy: 1.00

OR Gate
[0, 0] -> predicted 0 | expected 0 | Correct
[0, 1] -> predicted 1 | expected 1 | Correct
[1, 0] -> predicted 1 | expected 1 | Correct
[1, 1] -> predicted 1 | expected 1 | Correct
Truth table accuracy: 1.00



XOR Gate
[0, 0] -> predicted 0 | expected 0 | Correct
[0, 1] -> predicted 1 | expected 1 | Correct
[1, 0] -> predicted 1 | expected 1 | Correct
[1, 1] -> predicted 0 | expected 0 | Correct
Truth table accuracy: 1.00


## Final Summary

| Practical | Topic | Dataset / source | Main evaluation |
|---|---|---|---|
| 1 | Descriptive statistics | Iris sepal-length measurements | Mean, median, population SD, mode |
| 2 | DataFrame operations | Iris | `describe()`, mean, sample SD |
| 3 | Linear regression | Diabetes | Test MSE |
| 4 | Polynomial regression | Diabetes | Test MSE |
| 5 | Logistic regression | Breast Cancer Wisconsin | Accuracy, confusion matrix, report |
| 6 | Naive Bayes | Wine recognition | Accuracy, confusion matrix, report |
| 7 | k-NN | Iris | Per-sample correct/wrong, accuracy |
| 8 | ID3-style entropy tree | Iris | Accuracy, tree rules, new sample |
| 9 | CART Gini tree | Wine recognition | Accuracy, tree rules, new sample |
| 10 | Neural network logic gates | Standard truth tables | Output per input and gate accuracy |

**Interpretation:** Regression MSE values and classification scores are specific to the selected split and model configuration. They are measured by executing this notebook; they should not be interpreted as guaranteed performance on other splits or datasets.